# PCA compression experiments

- Three native conditions selected by observed CARec cold NDCG@10: Qwen without instruction 0.0352, MPNet 0.0326, E5-large without instruction 0.0321. Selection uses previous test results, so this is an exploratory compression study.
- Compare 64, 128 and 256 dimensions with the saved native references. Nine separate CARec runs, each followed by content-only evaluation. No new encoder inference.
- Fit centered PCA on unique products with training interactions only. No feature standardization or whitening. Fit 256 components once per condition; use their leading 64/128/256 components and L2-normalize each output.
- Preserve the existing split and repeated interactions. Native references keep their original training histories and budgets. Compression uses the settings below; changing dimension also changes CARec capacity.
- Run preparation first, then launch only the experiment cells you want. Run All will launch all nine experiments. No cells have been executed by Codex.

## Start here

1. Use the `NI-PML benchmarks` kernel. Run setup and the PCA function definitions.
2. Run the three `Prepare` cells. They save all nine compressed caches without training CARec.
3. Run a cell under `Train and evaluate`, starting with Qwen PCA 64. Each cell trains one CARec model and then evaluates content-only. Once the first run completes, run the other eight cells.
4. Run the comparison cell at any time to read available native/compressed results and export `compression_summary.csv`.


In [ ]:
from pathlib import Path
import hashlib
import json
import sys

import numpy as np
import pandas as pd
import sklearn
from sklearn.decomposition import PCA
from IPython.display import display

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "requirements.txt").is_file()
)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from helper import Experiment, RESULT_FILES

DATASET_ID = "electronics_carec_035_seed2023"
DIMENSIONS = (64, 128, 256)
SOURCES = {
    "qwen": "Qwen3-Embedding-0.6B-without_instruction.npz",
    "mpnet": "all-mpnet-base-v2.npz",
    "e5_large": "multilingual-e5-large-instruct-without_instruction.npz",
}

# Check inputs before loading the shared dataset or fitting PCA.
required_files = [
    PROJECT_ROOT / "data/processed" / DATASET_ID / name
    for name in ("manifest.json", "train.tsv.gz", "valid.tsv.gz", "test.tsv.gz", "cold.tsv.gz", "item_ids.tsv.gz")
]
required_files += [
    PROJECT_ROOT / "artifacts/embeddings" / DATASET_ID / name
    for name in SOURCES.values()
]
missing_files = [str(path) for path in required_files if not path.is_file()]
if missing_files:
    raise FileNotFoundError("Missing PCA inputs:\n" + "\n".join(missing_files))

experiment = Experiment(
    PROJECT_ROOT, DATASET_ID, device="mps", seed=2023,
    phase1_epochs=90, phase2_epochs=30, patience=15,
    training_batch_size=2048, learning_rate=0.001, weight_decay=0.0,
)
PCA_DIR = PROJECT_ROOT / "artifacts/pca" / DATASET_ID

## Prepare compressed caches

- PCA functions stay here because only this notebook uses them. Training and evaluation come from `helper.py`.
- Store the fitted mean, components, explained variance and training product IDs under `artifacts/pca/`. Compressed caches preserve native metadata and item order.
- Reuse matching files. Source/split hashes, solver settings and transform identity must match; conflicting files raise an error instead of overwriting them.
- [PCA documentation, scikit-learn 1.6](https://scikit-learn.org/1.6/modules/generated/sklearn.decomposition.PCA.html).

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as source:
        for block in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def save_npz(path, **arrays):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(".npz.tmp")
    with temporary.open("wb") as destination:
        np.savez(destination, **arrays)
    temporary.replace(path)


def prepare_pca_caches(embedding_name):
    source_path = experiment.embedding_dir / embedding_name
    with np.load(source_path, allow_pickle=False) as saved:
        native = {key: saved[key] for key in saved.files}
    item_ids = native["item_ids"]
    vectors = native.pop("embeddings").astype(np.float32)
    if vectors.ndim != 2 or len(vectors) != len(item_ids):
        raise ValueError("Expected one embedding per item ID.")
    rows = {product_id: row for row, product_id in enumerate(item_ids)}
    if len(rows) != len(item_ids) or set(item_ids) != set(experiment.recbole_product_ids[1:]):
        raise ValueError("Duplicate IDs or different product catalogues.")
    if not np.isfinite(vectors).all() or np.any(np.linalg.norm(vectors, axis=1) <= 1e-12):
        raise ValueError("Native vectors must be finite and nonzero.")

    train_ids = experiment.train_dataset.inter_feat[experiment.base_config["ITEM_ID_FIELD"]].unique().cpu().numpy()
    train_ids = np.sort(train_ids[train_ids != 0])
    fit_item_ids = np.asarray(experiment.recbole_product_ids[train_ids], dtype=str)
    fit_rows = np.asarray([rows[item_id] for item_id in fit_item_ids])
    if not DIMENSIONS or min(DIMENSIONS) < 1 or max(DIMENSIONS) >= min(len(fit_rows), vectors.shape[1]):
        raise ValueError("PCA dimensions must be positive and smaller than native size and fit count.")
    identity = {
        "schema_version": 1, "source_file": embedding_name,
        "source_sha256": sha256_file(source_path),
        "train_sha256": sha256_file(experiment.prepared_dir / "train.tsv.gz"),
        "fit_ids_sha256": hashlib.sha256(json.dumps(fit_item_ids.tolist()).encode()).hexdigest(),
        "fit_item_count": len(fit_rows), "max_dimension": max(DIMENSIONS),
        "solver": "full", "whiten": False, "standardize": False,
        "dtype": "float32", "sklearn_version": sklearn.__version__,
    }
    fit_path = PCA_DIR / f"{source_path.stem}-fit{max(DIMENSIONS)}.npz"
    if fit_path.exists():
        with np.load(fit_path, allow_pickle=False) as saved:
            if json.loads(str(saved["identity"])) != identity or not np.array_equal(saved["fit_item_ids"], fit_item_ids):
                raise ValueError(f"PCA provenance differs: {fit_path}")
            mean = saved["mean"]
            components = saved["components"]
            variance_ratio = saved["explained_variance_ratio"]
    else:
        print(f"Fitting PCA: {embedding_name}, {len(fit_rows):,} training items")
        pca = PCA(n_components=max(DIMENSIONS), svd_solver="full", whiten=False)
        pca.fit(vectors[fit_rows])
        mean = pca.mean_.astype(np.float32)
        components = pca.components_.astype(np.float32)
        variance_ratio = pca.explained_variance_ratio_
        save_npz(
            fit_path, identity=json.dumps(identity, sort_keys=True), fit_item_ids=fit_item_ids,
            mean=mean, components=components, explained_variance_ratio=variance_ratio,
        )
    if components.shape != (max(DIMENSIONS), vectors.shape[1]) or mean.shape != (vectors.shape[1],):
        raise ValueError("Invalid PCA transform shape.")
    if not np.isfinite(components).all() or not np.isfinite(mean).all():
        raise ValueError("Non-finite PCA transform.")

    if variance_ratio.shape != (max(DIMENSIONS),) or not np.isfinite(variance_ratio).all():
        raise ValueError("Invalid PCA explained variance.")

    filenames = {}
    preparation_rows = []
    fit_sha256 = sha256_file(fit_path)
    for dimension in DIMENSIONS:
        filename = f"{source_path.stem}-pca{dimension}.npz"
        destination = experiment.embedding_dir / filename
        provenance = {
            **identity, "dimension": dimension, "normalization": "l2_after_pca",
            "fit_sha256": fit_sha256,
            "fit_file": str(fit_path.relative_to(PROJECT_ROOT)),
        }
        if destination.exists():
            with np.load(destination, allow_pickle=False) as saved:
                if json.loads(str(saved["compression"])) != provenance or not np.array_equal(saved["item_ids"], item_ids):
                    raise ValueError(f"Compressed cache provenance differs: {destination}")
                compressed = saved["embeddings"]
        else:
            run_dir = experiment.runs_dir / Path(filename).stem
            if any(run_dir.glob("*.pt")):
                raise ValueError(f"Missing cache has existing checkpoints: {run_dir}")
            compressed = ((vectors - mean) @ components[:dimension].T).astype(np.float32)
        norms = np.linalg.norm(compressed, axis=1, keepdims=True)
        if compressed.shape != (len(item_ids), dimension) or not np.isfinite(compressed).all() or np.any(norms <= 1e-12):
            raise ValueError(f"Invalid compressed embeddings: {filename}")
        if destination.exists():
            if not np.allclose(norms, 1.0, atol=1e-5):
                raise ValueError(f"Cached vectors are not normalized: {filename}")
        else:
            compressed = compressed / norms
            save_npz(destination, **native, embeddings=compressed, compression=json.dumps(provenance, sort_keys=True))
            with np.load(destination, allow_pickle=False) as saved:
                if not np.array_equal(saved["item_ids"], item_ids) or not np.array_equal(saved["embeddings"], compressed):
                    raise ValueError(f"Compressed cache reload differs: {destination}")
        filenames[dimension] = filename
        preparation_rows.append({
            "file": filename, "dimension": dimension,
            "explained_variance": float(variance_ratio[:dimension].sum()),
            "cache_mb": destination.stat().st_size / 1e6,
            "vector_mb": compressed.nbytes / 1e6,
        })
    display(pd.DataFrame(preparation_rows))
    return filenames

### Prepare Qwen without instruction

In [ ]:
qwen_pca = prepare_pca_caches(SOURCES["qwen"])

### Prepare MPNet

In [ ]:
mpnet_pca = prepare_pca_caches(SOURCES["mpnet"])

### Prepare E5-large without instruction

In [ ]:
e5_large_pca = prepare_pca_caches(SOURCES["e5_large"])

## Train and evaluate

- Each cell launches one CARec condition, then evaluates content-only on the same compressed vectors. Keep separate result variables.
- The shared checkpoint guard still applies: existing phase two means evaluation only. With phase one alone, add `phase2_only=True` to start phase two from its best weights. This is not exact optimizer/epoch resume.
- Start with one condition. Other cells do not need to run in order after preparation.

### Qwen without instruction, PCA 64

In [ ]:
qwen_64_result = experiment.run_experiment(qwen_pca[64])
qwen_64_content_result = experiment.evaluate_content_only(qwen_pca[64])

### Qwen without instruction, PCA 128

In [ ]:
qwen_128_result = experiment.run_experiment(qwen_pca[128])
qwen_128_content_result = experiment.evaluate_content_only(qwen_pca[128])

### Qwen without instruction, PCA 256

In [ ]:
qwen_256_result = experiment.run_experiment(qwen_pca[256])
qwen_256_content_result = experiment.evaluate_content_only(qwen_pca[256])

### MPNet, PCA 64

In [ ]:
mpnet_64_result = experiment.run_experiment(mpnet_pca[64])
mpnet_64_content_result = experiment.evaluate_content_only(mpnet_pca[64])

### MPNet, PCA 128

In [ ]:
mpnet_128_result = experiment.run_experiment(mpnet_pca[128])
mpnet_128_content_result = experiment.evaluate_content_only(mpnet_pca[128])

### MPNet, PCA 256

In [ ]:
mpnet_256_result = experiment.run_experiment(mpnet_pca[256])
mpnet_256_content_result = experiment.evaluate_content_only(mpnet_pca[256])

### E5-large without instruction, PCA 64

In [ ]:
e5_large_64_result = experiment.run_experiment(e5_large_pca[64])
e5_large_64_content_result = experiment.evaluate_content_only(e5_large_pca[64])

### E5-large without instruction, PCA 128

In [ ]:
e5_large_128_result = experiment.run_experiment(e5_large_pca[128])
e5_large_128_content_result = experiment.evaluate_content_only(e5_large_pca[128])

### E5-large without instruction, PCA 256

In [ ]:
e5_large_256_result = experiment.run_experiment(e5_large_pca[256])
e5_large_256_content_result = experiment.evaluate_content_only(e5_large_pca[256])

## Compare with native references

- Read saved metrics without rerunning native models. Show available results only; missing method/condition pairs are listed separately.
- Cache MB includes IDs and metadata; vector MB is the float32 matrix only. Checkpoint MB counts selected phase-two weights, not both phase files. PCA transform MB is separate and shared across compressed dimensions.
- Compression keeps original caches and creates new files, so total experiment storage grows. These per-condition sizes describe the deployed representation and selected model.
- A single seed and the test-based shortlist do not establish statistical significance or generalization beyond these selected conditions.

In [ ]:
saved_results = {method: experiment.load_experiment_results(method) for method in RESULT_FILES}
comparison_rows = []
missing = []
for source in SOURCES.values():
    native_variant = Path(source).stem
    for dimension in (None, *DIMENSIONS):
        variant = native_variant if dimension is None else f"{native_variant}-pca{dimension}"
        for method, results in saved_results.items():
            if variant not in results:
                missing.append({"variant": variant, "method": method})
                continue
            result = results[variant]
            cache = experiment.embedding_dir / f"{variant}.npz"
            checkpoint = experiment.runs_dir / variant / "phase2-best.pt"
            fit = PCA_DIR / f"{native_variant}-fit{max(DIMENSIONS)}.npz"
            row = {
                "encoder": native_variant, "representation": "native" if dimension is None else "PCA",
                "variant": variant, "method": method, "dimension": result["dimension"],
                "cache_mb": cache.stat().st_size / 1e6 if cache.exists() else None,
                "vector_mb": (experiment.train_dataset.item_num - 1) * result["dimension"] * 4 / 1e6,
                "checkpoint_mb": checkpoint.stat().st_size / 1e6 if method == "CARec" and checkpoint.exists() else None,
                "pca_transform_mb": fit.stat().st_size / 1e6 if dimension is not None and fit.exists() else None,
            }
            for split_name, metrics in result["test"].items():
                row.update({f"{split_name}_{metric}": value for metric, value in metrics.items()})
            comparison_rows.append(row)
comparison = pd.DataFrame(comparison_rows)
display(comparison)
if missing:
    print("Not evaluated yet:")
    display(pd.DataFrame(missing))
if not comparison.empty:
    experiment.results_dir.mkdir(parents=True, exist_ok=True)
    temporary = experiment.results_dir / "compression_summary.csv.tmp"
    comparison.to_csv(temporary, index=False)
    temporary.replace(experiment.results_dir / "compression_summary.csv")